# 12 — MQA, GQA, and MLA

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

MHA stores separate K/V heads for every query head. This can make KV cache huge. MQA and GQA reduce KV heads; MLA compresses K/V into a latent representation.

Founder lens: these are inference-cost design choices. They affect serving memory, bandwidth, latency, and sometimes quality.

In [ ]:
def repeat_kv(x, n_rep):
    # x: [B, kv_heads, T, d]
    if n_rep == 1: return x
    B,H,T,D = x.shape
    return x[:, :, None, :, :].expand(B,H,n_rep,T,D).reshape(B,H*n_rep,T,D)

class AttentionVariant(nn.Module):
    def __init__(self, d_model=64, q_heads=8, kv_heads=8):
        super().__init__()
        assert q_heads % kv_heads == 0
        self.q_heads, self.kv_heads = q_heads, kv_heads
        self.d_head = d_model // q_heads
        self.q = nn.Linear(d_model, q_heads*self.d_head)
        self.k = nn.Linear(d_model, kv_heads*self.d_head)
        self.v = nn.Linear(d_model, kv_heads*self.d_head)
        self.o = nn.Linear(q_heads*self.d_head, d_model)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,self.q_heads,self.d_head).transpose(1,2)
        k = self.k(x).view(B,T,self.kv_heads,self.d_head).transpose(1,2)
        v = self.v(x).view(B,T,self.kv_heads,self.d_head).transpose(1,2)
        k = repeat_kv(k, self.q_heads // self.kv_heads)
        v = repeat_kv(v, self.q_heads // self.kv_heads)
        scores = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~torch.tril(torch.ones(T,T)).bool()[None,None,:,:], float('-inf'))
        y = torch.softmax(scores, dim=-1) @ v
        y = y.transpose(1,2).contiguous().view(B,T,self.q_heads*self.d_head)
        return self.o(y)

x = torch.randn(2,6,64)
for kv_heads, name in [(8,'MHA'), (1,'MQA'), (2,'GQA-2'), (4,'GQA-4')]:
    m = AttentionVariant(q_heads=8, kv_heads=kv_heads)
    print(name, m(x).shape, 'params', count_params(m))


In [ ]:
def kv_cache_gb(layers=32, q_heads=32, kv_heads=32, head_dim=128, seq_len=8192, batch=1, dtype_bytes=2):
    return batch * layers * 2 * kv_heads * seq_len * head_dim * dtype_bytes / 1e9

rows = []
for kv_heads, name in [(32,'MHA'), (1,'MQA'), (4,'GQA-4'), (8,'GQA-8')]:
    rows.append({'variant': name, 'kv_heads': kv_heads, 'GB_for_8k_context': kv_cache_gb(kv_heads=kv_heads)})
pd.DataFrame(rows)


## MLA toy sketch

Multi-head Latent Attention stores a compressed latent K/V state and reconstructs per-head K/V when needed. This cell is a teaching sketch, not a faithful DeepSeek implementation.

In [ ]:
class ToyMLA(nn.Module):
    def __init__(self, d_model=64, q_heads=8, latent_dim=16):
        super().__init__()
        self.q_heads = q_heads
        self.d_head = d_model // q_heads
        self.q = nn.Linear(d_model, q_heads*self.d_head)
        self.kv_down = nn.Linear(d_model, latent_dim)      # store this smaller latent cache
        self.k_up = nn.Linear(latent_dim, q_heads*self.d_head)
        self.v_up = nn.Linear(latent_dim, q_heads*self.d_head)
        self.o = nn.Linear(d_model, d_model)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,self.q_heads,self.d_head).transpose(1,2)
        latent = self.kv_down(x)
        k = self.k_up(latent).view(B,T,self.q_heads,self.d_head).transpose(1,2)
        v = self.v_up(latent).view(B,T,self.q_heads,self.d_head).transpose(1,2)
        scores = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        y = torch.softmax(scores.masked_fill(~mask[None,None,:,:], float('-inf')), dim=-1) @ v
        return self.o(y.transpose(1,2).contiguous().view(B,T,C)), latent

mla = ToyMLA()
y, latent_cache = mla(x)
print('output:', y.shape, 'stored latent cache:', latent_cache.shape)


## Decision notes

- MHA: strongest baseline, expensive KV cache.
- MQA: smallest KV cache, can lose quality if too aggressive.
- GQA: compromise between MHA and MQA.
- MLA: more complex compression strategy; attractive when decode memory bandwidth dominates.